In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

base = "/content/drive/MyDrive/Stripe_RAG_Project"

folders = [
    "data",
    "data/vector_db",
    "src",
    "notebooks",
    "outputs"
]

for folder in folders:
    os.makedirs(os.path.join(base, folder), exist_ok=True)

print("Folders Created Successfully")

Folders Created Successfully


In [ ]:
!pip install -q unsloth
!pip install -q transformers
!pip install -q accelerate
!pip install -q bitsandbytes
!pip install -q sentence-transformers
!pip install -q faiss-gpu
!pip install -q langchain
!pip install -q langchain-community
!pip install -q pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 755.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 104.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 127.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 121.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
!pip uninstall -y torchcodec
!pip install -q pypdf
!pip install -q langchain-community
!pip install -q langchain

Found existing installation: torchcodec 0.11.0+cu128
Uninstalling torchcodec-0.11.0+cu128:
  Successfully uninstalled torchcodec-0.11.0+cu128


In [ ]:
urls = [
    "https://docs.stripe.com/api.md",
    "https://docs.stripe.com/keys.md",
    "https://docs.stripe.com/checkout/quickstart.md",
    "https://docs.stripe.com/payments.md",
    "https://docs.stripe.com/refunds.md",
    "https://docs.stripe.com/billing/subscriptions/overview.md",
    "https://docs.stripe.com/webhooks.md",
    "https://docs.stripe.com/payments/payment-intents.md",
    "https://docs.stripe.com/api/payment_intents/confirm.md",
]

In [ ]:
import requests, os

save_dir = "/content/drive/MyDrive/Stripe_RAG_Project/data/raw_docs_md"
os.makedirs(save_dir, exist_ok=True)

for url in urls:
    path_part = url.replace("https://docs.stripe.com/", "").replace(".md", "")
    filename = path_part.replace("/", "_") + ".md"
    response = requests.get(url)
    if response.status_code == 200:
        with open(f"{save_dir}/{filename}", "w") as f:
            f.write(response.text)
        print(f"Saved: {filename} ({len(response.text)} chars)")
    else:
        print(f"FAILED ({response.status_code}): {url}")

Saved: api.md (1562 chars)
Saved: keys.md (18688 chars)
Saved: checkout_quickstart.md (37131 chars)
Saved: payments.md (5305 chars)
Saved: refunds.md (24443 chars)
Saved: billing_subscriptions_overview.md (16064 chars)
Saved: webhooks.md (48119 chars)
Saved: payments_payment-intents.md (15491 chars)
Saved: api_payment_intents_confirm.md (192028 chars)


In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(save_dir, glob="*.md", loader_cls=TextLoader)
documents = loader.load()
print(f"Loaded {len(documents)} markdown documents")

/tmp/ipykernel_2855/4101370549.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


Loaded 9 markdown documents


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks")

Created 464 chunks


In [ ]:
clean_chunks = []
for chunk in chunks:
    text = chunk.page_content.strip()
    if len(text) < 50:
        continue
    chunk.page_content = text
    clean_chunks.append(chunk)

print(f"Clean chunks: {len(clean_chunks)} (from {len(chunks)})")

Clean chunks: 464 (from 464)


In [ ]:
!pip install -q sentence-transformers
!pip install -q faiss-gpu
!pip install -q langchain-huggingface

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": device}
)

vector_db = FAISS.from_documents(clean_chunks, embeddings)
vector_db.save_local("/content/drive/MyDrive/Stripe_RAG_Project/data/vector_db")
print(f"New vector_db built and saved with {len(clean_chunks)} chunks")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

New vector_db built and saved with 464 chunks


In [ ]:
retriever = vector_db.as_retriever(search_kwargs={"k": 3})

docs = retriever.invoke("tell me about api")
for i, doc in enumerate(docs):
    print(f"--- Doc {i+1} ---")
    print(doc.metadata.get("source"))
    print(doc.page_content[:300])
    print()

--- Doc 1 ---
/content/drive/MyDrive/Stripe_RAG_Project/data/raw_docs_md/payments.md
- [API tour](https://docs.stripe.com/payments-api/tour.md)
- [Payment Intents](https://docs.stripe.com/payments/payment-intents.md)
- [Setup Intents](https://docs.stripe.com/payments/setup-intents.md)
- [Payment Methods](https://docs.stripe.com/payments/payment-methods.md)

- [Development environmen

--- Doc 2 ---
/content/drive/MyDrive/Stripe_RAG_Project/data/raw_docs_md/payments_payment-intents.md
Building an integration with the Payment Intents API involves two actions: creating and *confirming* (Confirming a PaymentIntent indicates that the customer intends to pay with the current or provided payment method. Upon confirmation, the PaymentIntent attempts to initiate a payment) a PaymentInten

--- Doc 3 ---
/content/drive/MyDrive/Stripe_RAG_Project/data/raw_docs_md/webhooks.md
#### Python

```python
org_api_key = os.environ.get("STRIPE_API_KEY")
webhook_secret = os.environ.get("WEBHOOK_SECRET")
clien

In [ ]:
!pip install -q langchain-text-splitters

In [ ]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings

device = "cuda" if torch.cuda.is_available() else "cpu"

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": device}
)

print(f"Embeddings ready on {device}!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings ready on cuda!


In [ ]:
from langchain_community.vectorstores import FAISS

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

# Switch model to fast inference mode (important for Unsloth speed gains)
FastLanguageModel.for_inference(model)

print("Model and tokenizer loaded successfully!")

/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:153: UserWarning: WARNING: Unsloth should be imported before [transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/3.83k [00:00<?, ?B/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.


Model and tokenizer loaded successfully!


In [ ]:
!pip install -q unsloth
!pip install -q transformers accelerate bitsandbytes

In [ ]:
%%writefile /content/drive/MyDrive/Stripe_RAG_Project/src/rag_pipeline.py

import os
import torch
from unsloth import FastLanguageModel
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# ==================================================
# Device
# ==================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# ==================================================
# Base Directory
# ==================================================
if os.path.exists("/content/drive/MyDrive/Stripe_RAG_Project"):
    BASE_DIR = "/content/drive/MyDrive/Stripe_RAG_Project"
else:
    BASE_DIR = os.path.dirname(
        os.path.dirname(os.path.abspath(__file__))
    )

VECTOR_DB_PATH = os.path.join(BASE_DIR, "data", "vector_db")

# ==================================================
# Load Embeddings
# ==================================================
print("Loading embeddings...")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": device},
)

# ==================================================
# Load FAISS
# ==================================================
print("Loading FAISS database...")

vector_db = FAISS.load_local(
    VECTOR_DB_PATH,
    embeddings,
    allow_dangerous_deserialization=True,
)

print("FAISS loaded successfully.")

# ==================================================
# Retriever
# ==================================================
retriever = vector_db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)


# ==================================================
# Load LLM
# ==================================================
print("Loading Llama model...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

FastLanguageModel.for_inference(model)

print("Model loaded successfully.")

# ==================================================
# Main Function
# ==================================================
def ask_stripe(question, chat_history=None):

    # Retrieve relevant chunks
    docs = retriever.invoke(question)

    context = "\n\n".join(
        [doc.page_content for doc in docs]
    )

    sources = list(
        set(
            doc.metadata.get("source", "unknown")
            for doc in docs
        )
    )

    system_prompt = """
You are a Stripe Documentation Assistant.

Answer ONLY using the information provided in the context.

Rules:
1. Do not use outside knowledge.
2. If the answer exists in the context, answer clearly and concisely.
3. If the context does not contain enough information, respond exactly with:

I could not find that information in the Stripe documentation.
"""

    user_prompt = f"""
Context:
{context}

Question:
{question}

Answer using only the context above.
"""

    messages = [
        {
            "role": "system",
            "content": system_prompt,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(device)

    attention_mask = torch.ones_like(inputs)

    with torch.no_grad():
        outputs = model.generate(
            inputs,
            attention_mask=attention_mask,
            max_new_tokens=400,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = outputs[0][inputs.shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    ).strip()

    return {
        "answer": response,
        "sources": sources,
    }

Overwriting /content/drive/MyDrive/Stripe_RAG_Project/src/rag_pipeline.py


In [ ]:
%run /content/drive/MyDrive/Stripe_RAG_Project/src/rag_pipeline.py

Using device: cuda
Loading embeddings...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading FAISS database...
FAISS loaded successfully.
Loading Llama model...
==((====))==  Unsloth 2026.6.9: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.


Model loaded successfully.


In [ ]:
%%writefile /content/drive/MyDrive/Stripe_RAG_Project/src/app.py
import streamlit as st
from rag_pipeline import ask_stripe

st.set_page_config(
    page_title="Stripe Documentation Assistant",
    page_icon="💳"
)

st.title("💳 Stripe Documentation Assistant")
st.write("Ask questions about Stripe documentation.")

if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

prompt = st.chat_input("Ask about Stripe...")

if prompt:
    st.session_state.messages.append(
        {"role": "user", "content": prompt}
    )

    with st.chat_message("user"):
        st.write(prompt)
    history = [msg["content"] for msg in st.session_state.messages if msg["role"] == "user"]
    result = ask_stripe(prompt, chat_history=history[:-1])
    answer = result["answer"]

    st.session_state.messages.append(
        {"role": "assistant", "content": answer}
    )

    with st.chat_message("assistant"):
        st.write(answer)

Overwriting /content/drive/MyDrive/Stripe_RAG_Project/src/app.py


In [ ]:
!pip install -q streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 91.7 MB/s eta 0:00:00


In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("3FpWkOdPKgallZpnF7Cs7OMyNVb_7rUJoPY7K3pQDdDMTkkLd")

In [ ]:
%cd /content/drive/MyDrive/Stripe_RAG_Project/src

/content/drive/MyDrive/Stripe_RAG_Project/src


In [ ]:
!streamlit run app.py &>/content/logs.txt &

In [ ]:
!cat /content/logs.txt



2026-07-02 14:34:25.341 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://8.228.23.6:8501



In [ ]:
public_url = ngrok.connect(8501)
print(public_url)

NgrokTunnel: "https://jam-flip-monkhood.ngrok-free.dev" -> "http://localhost:8501"


In [ ]:
!pkill -f streamlit
%cd /content/drive/MyDrive/Stripe_RAG_Project/src
!streamlit run app.py --server.port 8501 &>/content/logs.txt &

/content/drive/MyDrive/Stripe_RAG_Project/src
